# KronoDroid Tabular Model Training

This notebook trains representative tabular models on the KronoDroid dynamic behavioral profiles only. The goal is not to propose a new architecture, but to provide controlled model probes for the paper's temporal-drift evaluation.

Models: Logistic Regression, Random Forest, Extra Trees, XGBoost if available, and MLP. Saved artifacts are later applied unchanged to the AndroZoo temporal hold-out.

In [1]:
# Imports and configuration
from pathlib import Path
import json
import random
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, average_precision_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=UserWarning)

SEED = 42
TEST_SIZE = 0.15
CANDIDATE_DATA_DIRS = [Path('../krono_dataset'), Path('dynamic_ieee_paper/krono_dataset'), Path('krono_dataset')]
DATA_DIR = next((p for p in CANDIDATE_DATA_DIRS if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Could not locate krono_dataset. Run from project root or dynamic_ieee_paper/deneyler.')
RESULT_DIR = Path('dynamic_ieee_paper/deneyler/results') if Path('dynamic_ieee_paper/deneyler').exists() else Path('results')
RESULT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
print('Data directory:', DATA_DIR.resolve())

Data directory: /home/tan/GitHub/paper-kangal-dynamic/dynamic_ieee_paper/krono_dataset


In [2]:
# Load KronoDroid tabular behavioral profiles
malware_path = DATA_DIR / 'krono_malware.csv'
benign_path = DATA_DIR / 'krono_benign.csv'

df_mal = pd.read_csv(malware_path)
df_ben = pd.read_csv(benign_path, engine='python')
df = pd.concat([df_mal, df_ben], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'Malware: {len(df_mal):,}')
print(f'Benign : {len(df_ben):,}')
print(f'Total  : {len(df):,}')
print(f'Columns: {df.shape[1]}')
print(df['label'].value_counts())

Malware: 7,151
Benign : 7,197
Total  : 14,348
Columns: 168
label
benign     7197
malware    7151
Name: count, dtype: int64


In [3]:
# Feature matrix: 165 behavioral features, metadata excluded
META_COLS = ['package_name', 'label', 'timestamp']
feature_cols = [c for c in df.columns if c not in META_COLS]
X = df[feature_cols].copy()
y = (df['label'] == 'malware').astype(int).to_numpy()

print('Behavioral feature count:', len(feature_cols))
assert len(feature_cols) == 165, f'Expected 165 behavioral features, got {len(feature_cols)}'
assert not X.columns.duplicated().any()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)
print(f'Train: {len(X_train):,}  malware ratio={y_train.mean():.3f}')
print(f'Val  : {len(X_val):,}  malware ratio={y_val.mean():.3f}')

Behavioral feature count: 165
Train: 12,195  malware ratio=0.498
Val  : 2,153  malware ratio=0.498


In [4]:
# Preprocessing and model definitions
# Fit preprocessing only on KronoDroid train split.
preprocess_scaled = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler()),
])
preprocess_tree = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

models = {
    'LogisticRegression': Pipeline([
        ('prep', preprocess_scaled),
        ('clf', LogisticRegression(max_iter=3000, class_weight='balanced', n_jobs=-1, random_state=SEED)),
    ]),
    'RandomForest': Pipeline([
        ('prep', preprocess_tree),
        ('clf', RandomForestClassifier(n_estimators=500, class_weight='balanced_subsample', n_jobs=-1, random_state=SEED)),
    ]),
    'ExtraTrees': Pipeline([
        ('prep', preprocess_tree),
        ('clf', ExtraTreesClassifier(n_estimators=500, class_weight='balanced', n_jobs=-1, random_state=SEED)),
    ]),
    'MLP': Pipeline([
        ('prep', preprocess_scaled),
        ('clf', MLPClassifier(hidden_layer_sizes=(256, 128, 64), activation='relu', alpha=1e-4, batch_size=256, learning_rate_init=1e-3, max_iter=120, early_stopping=True, validation_fraction=0.12, n_iter_no_change=12, random_state=SEED)),
    ]),
}

try:
    from xgboost import XGBClassifier
    models['XGBoost'] = Pipeline([
        ('prep', preprocess_tree),
        ('clf', XGBClassifier(
            n_estimators=600, max_depth=5, learning_rate=0.04,
            subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            objective='binary:logistic', eval_metric='logloss',
            tree_method='hist', random_state=SEED, n_jobs=-1,
        )),
    ])
    print('XGBoost available: using XGBClassifier')
except Exception as exc:
    models['HistGradientBoosting'] = Pipeline([
        ('prep', preprocess_tree),
        ('clf', HistGradientBoostingClassifier(max_iter=400, learning_rate=0.04, l2_regularization=0.01, random_state=SEED)),
    ])
    print('XGBoost unavailable; using HistGradientBoosting fallback:', repr(exc))

print('Models:', ', '.join(models))

XGBoost available: using XGBClassifier
Models: LogisticRegression, RandomForest, ExtraTrees, MLP, XGBoost


In [5]:
# Train and evaluate models on KronoDroid validation split
def get_scores(model, X_eval):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X_eval)[:, 1]
    if hasattr(model, 'decision_function'):
        s = model.decision_function(X_eval)
        return (s - s.min()) / (s.max() - s.min() + 1e-12)
    return model.predict(X_eval)

rows = []
reports = {}
for name, model in models.items():
    print()
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    prob = get_scores(model, X_val)
    pred = (prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
    row = {
        'model': name,
        'accuracy': accuracy_score(y_val, pred),
        'macro_f1': f1_score(y_val, pred, average='macro'),
        'precision': precision_score(y_val, pred, zero_division=0),
        'recall': recall_score(y_val, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_val, prob),
        'pr_auc': average_precision_score(y_val, prob),
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
        'fpr': fp / (fp + tn + 1e-12),
        'fnr': fn / (fn + tp + 1e-12),
    }
    rows.append(row)
    reports[name] = classification_report(y_val, pred, target_names=['benign', 'malware'], output_dict=True)
    print(pd.Series(row).drop(['model']).round(4).to_string())

results = pd.DataFrame(rows).sort_values(['roc_auc', 'macro_f1'], ascending=False)
results.to_csv(RESULT_DIR / 'kronodroid_tabular_results.csv', index=False)
with open(RESULT_DIR / 'kronodroid_tabular_reports.json', 'w') as f:
    json.dump(reports, f, indent=2)
results


Training LogisticRegression...


/home/tan/anaconda3/envs/kangal/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


accuracy     0.8997
macro_f1     0.8995
precision    0.9307
recall        0.863
roc_auc      0.9365
pr_auc       0.9403
tn             1011
fp               69
fn              147
tp              926
fpr          0.0639
fnr           0.137

Training RandomForest...
accuracy     0.9545
macro_f1     0.9544
precision    0.9803
recall       0.9273
roc_auc      0.9806
pr_auc       0.9824
tn             1060
fp               20
fn               78
tp              995
fpr          0.0185
fnr          0.0727

Training ExtraTrees...
accuracy     0.9531
macro_f1     0.9531
precision    0.9737
recall        0.931
roc_auc      0.9779
pr_auc        0.981
tn             1053
fp               27
fn               74
tp              999
fpr           0.025
fnr           0.069

Training MLP...
accuracy     0.9322
macro_f1     0.9321
precision    0.9549
recall       0.9068
roc_auc      0.9549
pr_auc       0.9499
tn             1034
fp               46
fn              100
tp              973
fpr          

,model,accuracy,macro_f1,precision,recall,roc_auc,pr_auc,tn,fp,fn,tp,fpr,fnr
4,XGBoost,0.954482,0.954443,0.979351,0.928239,0.983469,0.987324,1059,21,77,996,0.019444,0.071761
1,RandomForest,0.954482,0.954441,0.980296,0.927307,0.980571,0.982353,1060,20,78,995,0.018519,0.072693
2,ExtraTrees,0.953089,0.953059,0.973684,0.931034,0.977922,0.981030,1053,27,74,999,0.025000,0.068966
3,MLP,0.932188,0.932133,0.954858,0.906803,0.954920,0.949856,1034,46,100,973,0.042593,0.093197
0,LogisticRegression,0.899675,0.899518,0.930653,0.863001,0.936463,0.940280,1011,69,147,926,0.063889,0.136999


In [6]:
# Optional: feature importance for tree-based probes
importance_rows = []
for name in ['RandomForest', 'ExtraTrees', 'XGBoost', 'HistGradientBoosting']:
    if name not in models:
        continue
    clf = models[name].named_steps['clf']
    if hasattr(clf, 'feature_importances_'):
        imp = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
        top = imp.head(25).reset_index()
        top.columns = ['feature', 'importance']
        top.insert(0, 'model', name)
        importance_rows.append(top)

if importance_rows:
    importance_df = pd.concat(importance_rows, ignore_index=True)
    importance_df.to_csv(RESULT_DIR / 'kronodroid_tabular_feature_importance.csv', index=False)
    display(importance_df.head(30))
else:
    print('No tree feature_importances_ available for the trained models.')

,model,feature,importance
0,RandomForest,session_duration_ms,0.090164
1,RandomForest,accessibility_query_count,0.074502
2,RandomForest,burst_peak_count,0.069624
3,RandomForest,getDeviceId_count,0.058746
4,RandomForest,surveillance_score,0.050761
5,RandomForest,anti_analysis_score_per_sec,0.038886
6,RandomForest,exfil_score,0.031948
7,RandomForest,stealth_score_per_sec,0.029793
8,RandomForest,implicit_intent_count,0.028343
9,RandomForest,anti_to_total_ratio,0.025611


In [7]:
# Export all trained tabular probes for frozen AndroZoo temporal inference
MODEL_DIR = RESULT_DIR / 'tabular_models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_files = {}
for name, model in models.items():
    model_path = MODEL_DIR / f'{name}.joblib'
    joblib.dump(model, model_path)
    model_files[name] = str(model_path.name)

manifest = {
    'source_dataset': 'KronoDroid',
    'task': 'tabular_temporal_inference',
    'label_mapping': {'benign': 0, 'malware': 1},
    'metadata_columns': META_COLS,
    'feature_count': len(feature_cols),
    'feature_columns': feature_cols,
    'models': model_files,
    'validation_results_file': 'kronodroid_tabular_results.csv',
}
with open(MODEL_DIR / 'tabular_model_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'Saved {len(model_files)} tabular models to {MODEL_DIR.resolve()}')
print(', '.join(model_files))

Saved 5 tabular models to /home/tan/GitHub/paper-kangal-dynamic/dynamic_ieee_paper/deneyler/results/tabular_models
LogisticRegression, RandomForest, ExtraTrees, MLP, XGBoost
